# RAID v2 end-to-end Colab runner

This notebook mounts Google Drive, clones the `jp_branch` branch, installs dependencies, downloads the datasets needed for Phases A-E, and then runs the matrix phase by phase. All outputs are written under `RAID_ARTIFACT_ROOT` on Drive so the run is resumable after Colab disconnects.

In [2]:
# === 1. Mount Drive and set artifact root ===
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

os.environ['RAID_ARTIFACT_ROOT'] = '/content/drive/MyDrive/raid_v2'
ARTIFACT_ROOT = Path(os.environ['RAID_ARTIFACT_ROOT'])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('RAID_ARTIFACT_ROOT =', ARTIFACT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAID_ARTIFACT_ROOT = /content/drive/MyDrive/raid_v2


In [3]:
# === 2. Clone or refresh the repo ===
REPO_URL = 'https://github.com/ConstantinVictorBeatErtel/RAID.git'
REPO_DIR = Path('/content/RAID')
BRANCH = 'jp_branch'

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd /content/RAID
    !git fetch origin
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}

%cd /content/RAID
!git status --short --branch
!ls v2 | head

Cloning into '/content/RAID'...
remote: Enumerating objects: 218, done.
remote: Counting objects: 100% (218/218), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 218 (delta 89), reused 200 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (218/218), 10.71 MiB | 18.18 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/RAID
## jp_branch...origin/jp_branch
configs
conftest.py
datasets
evaluate.py
features.py
heads
__init__.py
legacy
notebooks
README.md


In [4]:
# === 3. Install Python dependencies ===
%cd /content/RAID
!pip install -q -r /content/RAID/v2/requirements.txt

/content/RAID


In [5]:
# === 4. Inline RoboMimic downloader ===
# Mirrors the Stanford CDN into the exact Drive layout v2 expects.
#
# Files are written as:
#   <artifact_root>/data/robomimic/v1.5/<task>/<variant>/<modality>_v141.hdf5
#
# This matches the Colab flow that produced the working Phase A/B/C runs.

import sys
import time
import urllib.request

ROBOMIMIC_BASE = 'http://downloads.cs.stanford.edu/downloads/rt_benchmark'
ROBOMIMIC_ROOT = ARTIFACT_ROOT / 'data' / 'robomimic' / 'v1.5'

def _progress(block_num, block_size, total_size):
    if total_size <= 0:
        return
    pct = min(100, 100 * block_num * block_size / total_size)
    sys.stdout.write(f'\r  {pct:5.1f}%  ({block_num * block_size / 1e6:.1f} / {total_size / 1e6:.1f} MB)')
    sys.stdout.flush()

def fetch_robomimic(task: str, variant: str, modality: str = 'low_dim') -> Path:
    """Download one RoboMimic HDF5 to Drive at the layout v2 expects.

    Files are stored as <task>/<variant>/<modality>_v141.hdf5 so the
    v2 hdf5_path_for resolver finds them on the first candidate path.
    """
    src_name = 'low_dim.hdf5' if modality == 'low_dim' else 'image.hdf5'
    dst = ROBOMIMIC_ROOT / task / variant / f'{modality}_v141.hdf5'
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 100_000:
        print(f'[skip ] {task}/{variant}/{modality}  ({dst.stat().st_size / 1e6:.1f} MB)')
        return dst
    url = f'{ROBOMIMIC_BASE}/{task}/{variant}/{src_name}'
    tmp = dst.with_suffix(dst.suffix + '.tmp')
    print(f'[fetch] {url}')
    t0 = time.time()
    try:
        urllib.request.urlretrieve(url, tmp, reporthook=_progress)
        sys.stdout.write('\n')
        tmp.replace(dst)
    except Exception as exc:
        if tmp.exists():
            tmp.unlink()
        raise RuntimeError(f'failed to download {url}: {exc}')
    print(f'[done ] {dst}  ({dst.stat().st_size / 1e6:.1f} MB in {time.time() - t0:.1f}s)')
    return dst

# Phase A only needs lift/ph low_dim.
# Phase B uses the low_dim files below.
# Phase C/D use square/ph image.
for task, variant, modality in [
    ('lift', 'ph', 'low_dim'),
    ('lift', 'mh', 'low_dim'),
    ('can', 'ph', 'low_dim'),
    ('can', 'mh', 'low_dim'),
    ('square', 'ph', 'low_dim'),
    ('square', 'mh', 'low_dim'),
    ('transport', 'ph', 'low_dim'),
    ('transport', 'mh', 'low_dim'),
    ('square', 'ph', 'image'),
]:
    fetch_robomimic(task, variant, modality)

[skip ] lift/ph/low_dim  (18.5 MB)
[skip ] lift/mh/low_dim  (47.9 MB)
[skip ] can/ph/low_dim  (45.5 MB)
[skip ] can/mh/low_dim  (112.5 MB)
[skip ] square/ph/low_dim  (50.0 MB)
[skip ] square/mh/low_dim  (123.9 MB)
[skip ] transport/ph/low_dim  (309.0 MB)
[skip ] transport/mh/low_dim  (637.0 MB)
[skip ] square/ph/image  (2603.4 MB)


In [6]:
# === 5. Verify the files are really on Drive before training ===
from pathlib import Path

root = Path('/content/drive/MyDrive/raid_v2/data/robomimic/v1.5')
checks = [
    root / 'lift/ph/low_dim_v141.hdf5',
    root / 'lift/mh/low_dim_v141.hdf5',
    root / 'can/ph/low_dim_v141.hdf5',
    root / 'can/mh/low_dim_v141.hdf5',
    root / 'square/ph/low_dim_v141.hdf5',
    root / 'square/mh/low_dim_v141.hdf5',
    root / 'transport/ph/low_dim_v141.hdf5',
    root / 'transport/mh/low_dim_v141.hdf5',
    root / 'square/ph/image_v141.hdf5',
]

missing = []
for p in checks:
    ok = p.exists()
    print(p, ok, f'{p.stat().st_size/1e6:.1f} MB' if ok else 'MISSING')
    if not ok:
        missing.append(str(p))

if missing:
    raise RuntimeError('Missing RoboMimic files on Drive:\n' + '\n'.join(missing))


/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/lift/ph/low_dim_v141.hdf5 True 18.5 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/lift/mh/low_dim_v141.hdf5 True 47.9 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/can/ph/low_dim_v141.hdf5 True 45.5 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/can/mh/low_dim_v141.hdf5 True 112.5 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/square/ph/low_dim_v141.hdf5 True 50.0 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/square/mh/low_dim_v141.hdf5 True 123.9 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/transport/ph/low_dim_v141.hdf5 True 309.0 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/transport/mh/low_dim_v141.hdf5 True 637.0 MB
/content/drive/MyDrive/raid_v2/data/robomimic/v1.5/square/ph/image_v141.hdf5 True 2603.4 MB


In [7]:
# === 6. GPU sanity check ===
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

GPU: True NVIDIA L4


## Phase A

Reproduce the low-dim Lift baseline in the new harness.

In [8]:
# === 6. Run Phase A ===
%cd /content/RAID
!python3 -m v2.run_matrix --phase A

/content/RAID
[run_matrix] expanded 36 cells (phase filter: A)
[run_matrix] [1/36] SKIP completed run_id=15df78d75b0ee339
[run_matrix] [2/36] SKIP completed run_id=377ba3088a0d2afa
[run_matrix] [3/36] SKIP completed run_id=09bf871c18233c17
[run_matrix] [4/36] SKIP completed run_id=ad6409dd573c0def
[run_matrix] [5/36] SKIP completed run_id=a2f505b7852a37a4
[run_matrix] [6/36] SKIP completed run_id=aeb2e5ca987bb964
[run_matrix] [7/36] SKIP completed run_id=7907b99cf3b1681c
[run_matrix] [8/36] SKIP completed run_id=c01ca7007b7764fc
[run_matrix] [9/36] SKIP completed run_id=29eac130d670c4c1
[run_matrix] [10/36] SKIP completed run_id=9081046a4d799a4e
[run_matrix] [11/36] SKIP completed run_id=5add2f1c6cc8527b
[run_matrix] [12/36] SKIP completed run_id=64ebb02f21e2f744
[run_matrix] [13/36] SKIP completed run_id=c06bd23c4cec9148
[run_matrix] [14/36] SKIP completed run_id=b7f40b00eb831070
[run_matrix] [15/36] SKIP completed run_id=22e874d8ff69e3f1
[run_matrix] [16/36] SKIP completed run_id=95c

## Phase B

Cross-dataset low-dim runs on the mixed RoboMimic tasks.

In [9]:
# === 7. Refresh repo before later phases ===
%cd /content/RAID
!git fetch origin
!git checkout jp_branch
!git reset --hard origin/jp_branch

/content/RAID
Already on 'jp_branch'
Your branch is up to date with 'origin/jp_branch'.
HEAD is now at 232a936 v2: align colab notebook with inline robomimic downloads


In [10]:
# === 8. Run Phase B ===
%cd /content/RAID
!python3 -m v2.run_matrix --phase B

/content/RAID
[run_matrix] expanded 18 cells (phase filter: B)
[run_matrix] [1/18] SKIP completed run_id=f06379bf716fd7a7
[run_matrix] [2/18] SKIP completed run_id=80acd1e41b5b85d3
[run_matrix] [3/18] SKIP completed run_id=fab815f0f2c9c6ca
[run_matrix] [4/18] SKIP completed run_id=13df8e9f375ab0bc
[run_matrix] [5/18] SKIP completed run_id=98fd4a0b3b009c80
[run_matrix] [6/18] SKIP completed run_id=1f34b6f0cacff221
[run_matrix] [7/18] SKIP completed run_id=38dd273a04bd7f1c
[run_matrix] [8/18] SKIP completed run_id=fad572795e2076a8
[run_matrix] [9/18] SKIP completed run_id=3d347d9236d7607f
[run_matrix] [10/18] SKIP completed run_id=a2e6e8c13507099c
[run_matrix] [11/18] SKIP completed run_id=5dea093d8d8b48ba
[run_matrix] [12/18] SKIP completed run_id=b045b66f45792875
[run_matrix] [13/18] SKIP completed run_id=2b5fc88800585a8a
[run_matrix] [14/18] SKIP completed run_id=8b8df20387dd74e4
[run_matrix] [15/18] SKIP completed run_id=6c357ffa9a4b2783
[run_matrix] [16/18] SKIP completed run_id=e24

## Phase C

Cache DINOv2 image features for `robomimic_square_ph_image`, then run the image-feature matrix.

In [11]:
# === 9. Extract DINOv2 features and run Phase C ===
%cd /content/RAID
!python3 -m v2.features --encoders dinov2 --datasets robomimic_square_ph_image
!python3 -m v2.run_matrix --phase C

/content/RAID
config.json: 100% 548/548 [00:00<00:00, 2.84MB/s]
model.safetensors: 100% 346M/346M [00:03<00:00, 107MB/s] 
Loading weights: 100% 223/223 [00:00<00:00, 1140.12it/s, Materializing param=layernorm.weight]                                
preprocessor_config.json: 100% 436/436 [00:00<00:00, 2.41MB/s]
The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[features] SKIP /content/drive/MyDrive/raid_v2/features/robomimic_square_ph_image_dinov2_cls.safetensors (already exists, 46.3 MB)
[run_matrix] expanded 24 cells (phase filter: C)
[run_matrix] [1/24] SKIP completed run_id=3a7d865e113f5541
[run_matrix] [2/24] SKIP completed run_id=d251e8701564a4c9
[run_matrix] [3/24] SKIP completed run_id=a28be9cfa72943ea
[run_matrix] [4/24] 

## Phase D

Encoder ablation: re-extract image features with **SigLIP** (semantic, language-aligned) and compare against the DINOv2 (spatial, self-supervised) results from Phase C using the same Transformer head. SigLIP loads via standard `transformers` without any monkey-patching.


In [ ]:
# === 10. Extract SigLIP features (encoder ablation) and run Phase D ===
%cd /content/RAID
!python3 -m v2.features --encoders siglip --datasets robomimic_square_ph_image
!python3 -m v2.run_matrix --phase D


## Phase E (optional)

Multi-task retrieval test on LIBERO-Goal. The `openvla/modified_libero_rlds` repo is TFDS-format and the HF `datasets` library cannot read it directly, so this phase only runs if you have a compatible LIBERO mirror staged on Drive. If feature extraction fails, the matrix runner reports the cells as `FAIL` and the rest of the notebook continues.


In [ ]:
# === 11. Try Phase E (LIBERO multi-task retrieval). Non-fatal on failure ===
%cd /content/RAID
import subprocess

def _try(cmd: list[str]) -> int:
    print('+ ' + ' '.join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0:
        print(f'[notebook] {cmd[2:]} exited with {r.returncode}; continuing.')
    return r.returncode

_try(['python3', '-m', 'v2.runtime.data_download', '--include-libero',
      '--tasks', 'lift', '--variants', 'ph', '--modalities', 'low_dim'])
_try(['python3', '-m', 'v2.features', '--encoders', 'dinov2', '--datasets', 'libero_goal'])
_try(['python3', '-m', 'v2.run_matrix', '--phase', 'E'])


## Aggregate results

Build the forest plot from all completed runs. This script must be run as a module.

In [ ]:
# === 12. Aggregate completed runs into summary figures ===
%cd /content/RAID
!python3 -m v2.notebooks.03_matrix_results

: 

In [ ]:
# === 13. Inspect generated figure paths ===
from pathlib import Path

fig_root = Path('/content/drive/MyDrive/raid_v2/results/figures')
for path in [fig_root / 'matrix_forest.png']:
    print(path, path.exists(), f'{path.stat().st_size / 1e6:.2f} MB' if path.exists() else '')

pred_root = fig_root / 'predictions'
if pred_root.exists():
    grids = sorted(pred_root.glob('*/grid.png'))
    print('prediction grids:', len(grids))
    for p in grids[-10:]:
        print(' ', p)

: 

In [ ]:
# === 14. Optional: dry-run resume checks ===
%cd /content/RAID
!python3 -m v2.run_matrix --phase A --dry-run
!python3 -m v2.run_matrix --phase B --dry-run
!python3 -m v2.run_matrix --phase C --dry-run
!python3 -m v2.run_matrix --phase D --dry-run
!python3 -m v2.run_matrix --phase E --dry-run

: 